# Session 1 · Introduction to Big Data and the Hadoop Ecosystem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Diploma-in-Data-Science/blob/main/term-2/02-big-data-systems-with-hadoop/code/session-01-big-data-and-hadoop-ecosystem.ipynb)

Diploma in Data Science · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run the cells from top to bottom. Every example is self-contained, so you can also run any single cell on its own.

### Example 1 · Why scale out? Reading 10 TB

In [1]:
data_tb = 10
disk_mb_per_s = 150            # typical sequential read speed of one hard disk
data_mb = data_tb * 1024 * 1024

print(f"{'machines':>8} | {'read time':>12}")
for machines in [1, 10, 50, 100, 500]:
    seconds = data_mb / (disk_mb_per_s * machines)
    h, rem = divmod(seconds, 3600)
    m = rem / 60
    print(f"{machines:>8} | {int(h):>3} h {m:5.1f} min")

machines |    read time
       1 |  19 h  25.1 min
      10 |   1 h  56.5 min
      50 |   0 h  23.3 min
     100 |   0 h  11.7 min
     500 |   0 h   2.3 min


### Example 2 · How HDFS splits a file into blocks

In [2]:
import math, random
random.seed(7)

BLOCK_MB, REPLICATION = 128, 3
datanodes = ["dn1", "dn2", "dn3", "dn4", "dn5"]

def split_file(name, size_mb):
    n = math.ceil(size_mb / BLOCK_MB)
    blocks = []
    for i in range(n):
        size = min(BLOCK_MB, size_mb - i * BLOCK_MB)
        replicas = random.sample(datanodes, REPLICATION)
        blocks.append((f"{name}#blk_{i+1}", size, replicas))
    return blocks

for fname, size in [("calls_jan.csv", 300), ("logo.png", 2), ("cctv_day1.mp4", 1000)]:
    blocks = split_file(fname, size)
    print(f"{fname}: {size} MB -> {len(blocks)} block(s), raw storage used = {size*REPLICATION} MB")
    for b, s, reps in blocks:
        print(f"   {b:<22} {s:>4} MB  on {', '.join(reps)}")

calls_jan.csv: 300 MB -> 3 block(s), raw storage used = 900 MB
   calls_jan.csv#blk_1     128 MB  on dn3, dn2, dn4
   calls_jan.csv#blk_2     128 MB  on dn1, dn5, dn3
   calls_jan.csv#blk_3      44 MB  on dn1, dn3, dn4
logo.png: 2 MB -> 1 block(s), raw storage used = 6 MB
   logo.png#blk_1            2 MB  on dn1, dn2, dn5
cctv_day1.mp4: 1000 MB -> 8 block(s), raw storage used = 3000 MB
   cctv_day1.mp4#blk_1     128 MB  on dn1, dn4, dn2
   cctv_day1.mp4#blk_2     128 MB  on dn1, dn2, dn5
   cctv_day1.mp4#blk_3     128 MB  on dn5, dn4, dn1
   cctv_day1.mp4#blk_4     128 MB  on dn5, dn1, dn4
   cctv_day1.mp4#blk_5     128 MB  on dn5, dn1, dn3
   cctv_day1.mp4#blk_6     128 MB  on dn5, dn4, dn1
   cctv_day1.mp4#blk_7     128 MB  on dn2, dn1, dn3
   cctv_day1.mp4#blk_8     104 MB  on dn2, dn3, dn5


### Example 3 · The MapReduce idea in plain Python

In [3]:
from collections import defaultdict

splits = [                                   # imagine each line is a block on a different node
    "big data needs big clusters",
    "hadoop stores big data in hdfs",
    "yarn shares the cluster",
]

def mapper(line):
    for word in line.split():
        yield word, 1

def reducer(word, counts):
    return word, sum(counts)

# map phase (runs in parallel on the nodes that hold each split)
mapped = [pair for line in splits for pair in mapper(line)]
print("map output (first 6):", mapped[:6])

# shuffle and sort: group all values by key
groups = defaultdict(list)
for k, v in mapped:
    groups[k].append(v)

# reduce phase
result = [reducer(k, v) for k, v in sorted(groups.items())]
print("reduce output:")
for word, n in sorted(result, key=lambda x: (-x[1], x[0])):
    print(f"  {word:<9}{n}")

map output (first 6): [('big', 1), ('data', 1), ('needs', 1), ('big', 1), ('clusters', 1), ('hadoop', 1)]
reduce output:
  big      3
  data     2
  cluster  1
  clusters 1
  hadoop   1
  hdfs     1
  in       1
  needs    1
  shares   1
  stores   1
  the      1
  yarn     1
